## Technology Parameters Dataset

Create `technologies_2030.csv` combining data from multiple sources:

**Data Sources:**
- **Capacity data**: `pemmdb_capacities_2030_grouped.csv` (69 unique `index_carrier` technologies)
- **Cost parameters**: `costs_2030_processed.csv` (efficiency, VOM, FOM from PyPSA technology-data)
- **Fuel prices**: TYNDP 2024 Study values (€/GJ → €/MWh via ×3.6)
- **CO2 emissions**: TYNDP 2024 Study values (kgCO2/GJ → tCO2/MWh via ×3.6/1000)

**Dataset Structure:**
- `index_carrier`: Specific technology from PEMMDB (69 types)
- `carrier`: Generic technology group (types: chp, gas, coal, onwind, etc.)
- `pypsa_carrier`: Network carrier for PyPSA (types: separates renewables, groups fossils by emission profile)
- `fuel_type`: Base fuel being burned (for fuel prices: coal, gas, lignite, oil-*, h2, nuclear, renewable)
- `emission_type`: Emission profile (for CO2: separates CCS variants like coal-ccs, gas-ccs)
- `costs_technology`: Technology name in costs CSV for parameter lookup

**Key TYNDP 2024 Study Values:**
- **Fuel prices (€/GJ)**: Nuclear 1.7, Coal 1.8, Gas 6.3, Oil-heavy 9.6, Oil-light 11.7, H2 17.6
- **CO2 emissions (kgCO2/GJ)**: 
  - Gas CCS: **5.70** (90% capture)
  - Gas (all non-CCS): **51.57**
  - Coal: 94.0, Coal-CCS: 9.4
  - Lignite: 101.0, Lignite-CCS: 10.1
- **CO2 price**: 113.4 €/tCO2

**Marginal Cost Calculation:**
```
MC = VOM + (fuel_price / efficiency) + (CO2_emissions / efficiency) × CO2_price
```

**CCS Technology Handling:**
- `fuel_type` = base fuel (e.g., 'coal' for coal-ccs) → used for fuel prices
- `emission_type` = technology variant (e.g., 'coal-ccs') → used for CO2 emissions
- `pypsa_carrier` = emission_type (to distinguish CCS from non-CCS in network)

In [15]:
import pandas as pd
import numpy as np

capacities = pd.read_csv('../data/open-tyndp/pemmdb_capacities_2030_grouped.csv')
costs = pd.read_csv('../data/open-tyndp/costs_2030_processed.csv')

In [16]:
print(capacities["index_carrier"].unique())

['coal' 'coal-ccs' 'gas-ccgt' 'gas-ccgt-ccs' 'gas-conv' 'gas-ocgt'
 'lignite' 'lignite-ccs' 'nuclear' 'oil-heavy' 'oil-light' 'oil-shale'
 'electrolyser-onshore-grid' 'hydro-ror-turbine' 'hydro-pondage-reservoir'
 'hydro-pondage-turbine' 'hydro-reservoir-reservoir'
 'hydro-reservoir-turbine' 'hydro-phs-reservoir' 'hydro-phs-turbine'
 'hydro-phs-pump' 'hydro-phs-pure-reservoir' 'hydro-phs-pure-turbine'
 'hydro-phs-pure-pump' 'onwind' 'offwind' 'solar-thermal'
 'solar-pv-utility' 'solar-pv-rooftop' 'solar-thermal-w-storage'
 'solar-thermal-w-storage-storage' 'battery-charge' 'battery-discharge'
 'battery-store' 'h2-fuel-cell' 'h2-ccgt' 'other-res'
 'chp-gas-ccgt-old-1-other-70.65eur'
 'chp-gas-conventional-old-2-industrial-0eur'
 'chp-gas-conventional-old-1-industrial-0eur'
 'chp-gas-conventional-old-1-other-0eur' 'chp-lignite-old-1-other-0eur'
 'chp-gas-conventional-old-1-other-128.4eur'
 'chp-gas-ccgt-old-2-other-58.87500000000006eur'
 'chp-gas-ccgt-old-1-other-0eur' 'chp-gas-conventio

In [17]:
print(capacities["carrier"].unique())

['coal' 'gas' 'lignite' 'uranium' 'oil-heavy' 'oil-light' 'oil-shale'
 'electrolyser' 'hydro-ror' 'hydro-pondage' 'hydro-reservoir' 'hydro-phs'
 'hydro-phs-pure' 'onwind' 'offwind' 'solar-thermal' 'solar-pv-utility'
 'solar-pv-rooftop' 'solar-thermal-w-storage' 'battery' 'h2-fuel-cell'
 'h2-ccgt' 'other-res' 'chp']


In [18]:
# Get unique index_carriers with their carrier mapping
carrier_mapping = capacities[['index_carrier', 'carrier']].drop_duplicates().set_index('index_carrier')['carrier'].to_dict()
index_carriers = sorted(capacities['index_carrier'].unique())
print(f"Found {len(index_carriers)} unique index_carriers\n")

# TYNDP 2024 Study Data
study_fuel_prices = {'nuclear': 1.7, 'coal': 1.8, 'lignite': 1.8, 'gas': 6.3, 
                     'oil-heavy': 9.6, 'oil-light': 11.7, 'oil-shale': 1.9, 'h2': 17.6}

# CO2 emissions - includes both base fuels and CCS technologies
study_co2_emissions = {'nuclear': 0.0, 'coal': 94.0, 'coal-ccs': 9.4, 'lignite': 101.0, 
                       'lignite-ccs': 10.1, 'gas': 51.57, 'gas-ccs': 5.70,
                       'oil-light': 78.0, 'oil-heavy': 78.0, 'oil-shale': 100.0, 'h2': 0.0}

co2_price = 113.4  # €/tCO2

Found 69 unique index_carriers



In [19]:

# Add pypsa_carrier column based on fuel type and emission type
# Strategy:
# - For CHP: Use 'other-thermal' (combined heat and power plants)
# - For renewables: Use the 'carrier' column (onwind, solar-pv-utility, battery, etc.)
# - For fossil/fuel-based: Use the 'emission_type' (gas, coal, gas-ccs, coal-ccs, etc.)
# - Special cases: h2-ccgt and h2-fuel-cell keep their own index_carrier as pypsa_carrier

def get_pypsa_carrier(row):
    """Determine PyPSA carrier name for network definition"""
    # Special case: hydrogen technologies keep their specific carrier
    if row['index_carrier'] in ('h2-ccgt', 'h2-fuel-cell'):
        return row['index_carrier']
    if row['carrier'] == 'chp':
        return 'other-thermal'  # All CHP plants grouped as other-thermal
    elif row['fuel_type'] == 'renewable':
        return row['carrier']  # Use carrier: onwind, offwind, solar-pv-utility, battery, etc.
    else:
        return row['emission_type']  # Use emission_type: gas, coal, gas-ccs, coal-ccs, etc.


# mapping function
def get_params(idx_c):
    """Map index_carrier to (base_fuel_type, emission_type, costs_technology)
    
    Returns:
        base_fuel: The actual fuel being burned (for fuel prices)
        emission_type: The emission profile/technology variant (for CO2 emissions, pypsa_carrier)
        costs_tech: Technology name in costs CSV
    """
    # Fossil fuels with CCS - separate base fuel from technology
    if 'coal-ccs' in idx_c or 'hard-coal-ccs' in idx_c: 
        return ('coal', 'coal-ccs', 'coal-ccs')  # Burns coal, but CCS technology
    if 'lignite-ccs' in idx_c: 
        return ('lignite', 'lignite-ccs', 'lignite-ccs')  # Burns lignite, but CCS technology
    if 'gas-ccgt-ccs' in idx_c: 
        return ('gas', 'gas-ccs', 'gas-ccgt-ccs')  # Burns gas, but CCS technology (use gas-ccgt-ccs for costs lookup)
    
    # Fossil fuels without CCS - base fuel = technology type
    if 'nuclear' in idx_c: return ('nuclear', 'nuclear', 'nuclear')
    if 'coal' in idx_c or 'hard-coal' in idx_c: return ('coal', 'coal', 'coal')
    if 'lignite' in idx_c: return ('lignite', 'lignite', 'lignite')
    if 'gas-ccgt' in idx_c or 'gas-conv' in idx_c: return ('gas', 'gas', 'CCGT')
    if 'gas-ocgt' in idx_c: return ('gas', 'gas', 'OCGT')
    
    # Oil variants
    if 'oil-light' in idx_c or 'light-oil' in idx_c: return ('oil-light', 'oil-light', 'oil-light')
    if 'oil-heavy' in idx_c or 'heavy-oil' in idx_c: return ('oil-heavy', 'oil-heavy', 'oil-heavy')
    if 'oil-shale' in idx_c: return ('oil-shale', 'oil-shale', 'oil-shale')
    
    # Hydrogen — keep specific index_carrier as emission_type so pypsa_carrier is set correctly
    if 'h2-ccgt' in idx_c: return ('h2', 'h2-ccgt', 'central hydrogen CHP')
    if 'h2-fuel-cell' in idx_c: return ('h2', 'h2-fuel-cell', 'fuel cell')
    
    # Renewables - no fuel, technology = pypsa carrier
    if 'onwind' in idx_c: return (None, None, 'onwind')
    if 'offwind' in idx_c: return (None, None, 'offwind')
    if 'solar-pv-utility' in idx_c: return (None, None, 'solar')
    if 'solar-pv-rooftop' in idx_c: return (None, None, 'solar-rooftop')
    if 'solar-thermal' in idx_c: return (None, None, 'central solar thermal')
    if 'other-res' in idx_c: return (None, None, None)  # Generic renewable
    
    # Hydro (all variants)
    if 'hydro' in idx_c: return (None, None, 'hydro')
    
    # Storage - Battery components need special handling
    if 'battery-charge' in idx_c or 'battery-discharge' in idx_c:
        return (None, None, 'battery inverter')  # Charge/discharge use inverter
    if 'battery-store' in idx_c or 'battery' in idx_c:
        return (None, None, 'battery storage')  # Storage vessel
    if 'electrolyser' in idx_c: return (None, None, 'electrolysis')
    
    return (None, None, None)

# Hardcoded marginal costs for specific carriers (not derivable from fuel/cost data)
HARDCODED_MC = {
    'other-res': 45.0,  # Biomass/waste opportunity cost — not a fuel price, set explicitly
}

# Build dataset
data = []
missing = []

for idx_c in index_carriers:
    base_fuel, tech_type, cost_tech = get_params(idx_c)
    carrier = carrier_mapping.get(idx_c, 'unknown')
    
    # Special handling for storage components (reservoir, store, pump)
    is_storage = any(x in idx_c for x in ['reservoir', 'store'])
    is_pump = 'pump' in idx_c
    
    # Look up from costs CSV
    if cost_tech:
        costs_row = costs[costs['technology'] == cost_tech]
        if not costs_row.empty:
            eff = costs_row['efficiency'].iloc[0]
            vom = costs_row['VOM'].iloc[0]
            fom = costs_row['FOM'].iloc[0]
        else:
            # Fallback for missing technologies
            if is_storage:
                eff, vom, fom = 1.0, 0.0, 0.0  # Storage: no conversion loss
            elif is_pump:
                eff, vom, fom = 0.9, 0.0, 1.0  # Pumps: ~90% efficiency
            else:
                missing.append(idx_c)
                eff, vom, fom = 1.0, 0.0, 0.0
    else:
        # No cost_tech specified
        eff, vom, fom = 1.0, 0.0, 0.0
    
    # Calculate marginal cost
    if base_fuel:  # Thermal generator with fuel
        fuel_price_gj = study_fuel_prices.get(base_fuel, 0)  # Use BASE fuel for price
        fuel_price_mwh = fuel_price_gj * 3.6
        
        # Use EMISSION type for CO2 (CCS has lower emissions)
        co2_kg_gj = study_co2_emissions.get(tech_type, 0)
        co2_tco2_mwh = co2_kg_gj * 3.6 / 1000
        
        if eff > 0:
            fuel_cost = fuel_price_mwh / eff
            co2_cost = (co2_tco2_mwh / eff) * co2_price
            mc = vom + fuel_cost + co2_cost
        else:
            mc = vom
    else:  # Renewable or storage
        fuel_price_gj, fuel_price_mwh = 0, 0
        co2_kg_gj, co2_tco2_mwh = 0, 0
        mc = vom

    # Override with hardcoded MC if defined for this carrier
    if carrier in HARDCODED_MC:
        mc = HARDCODED_MC[carrier]
    
    data.append({
        'index_carrier': idx_c,
        'carrier': carrier,
        'fuel_type': base_fuel if base_fuel else 'renewable',  # Base fuel for fuel prices
        'emission_type': tech_type if tech_type else 'renewable',  # Emission profile (includes CCS variants)
        'costs_technology': cost_tech if cost_tech else '',
        'efficiency': eff if pd.notna(eff) else 1.0,
        'vom_eur_mwh': vom if pd.notna(vom) else 0.0,
        'fom_eur_mw_year': fom if pd.notna(fom) else 0.0,
        'fuel_price_eur_gj': fuel_price_gj,
        'fuel_price_eur_mwh': fuel_price_mwh,
        'co2_kg_gj': co2_kg_gj,
        'co2_tco2_mwh': co2_tco2_mwh,
        'marginal_cost_eur_mwh': mc if pd.notna(mc) else 0.0,
    })

df = pd.DataFrame(data)

df['pypsa_carrier'] = df.apply(get_pypsa_carrier, axis=1)

# Reorder columns
cols = ['index_carrier', 'carrier', 'pypsa_carrier', 'fuel_type', 'emission_type', 
        'costs_technology', 'efficiency', 'vom_eur_mwh', 'fom_eur_mw_year', 
        'fuel_price_eur_gj', 'fuel_price_eur_mwh', 'co2_kg_gj', 'co2_tco2_mwh', 
        'marginal_cost_eur_mwh']
df = df[cols]

df.to_csv('../data/open-tyndp/technologies_2030.csv', index=False)

print(f"Created dataset with {len(df)} technologies")
print(f"{len(missing)} technologies not found in costs CSV")
if missing:
    print(f"  Missing: {missing}\n")

print("\nUnique PyPSA carriers:")
unique_carriers = sorted(df['pypsa_carrier'].unique())
print(f"Total: {len(unique_carriers)} carriers")
print(unique_carriers)

# Verify hardcoded MC values
print("\nHardcoded MC verification:")
for carrier_name, expected_mc in HARDCODED_MC.items():
    rows = df[df['carrier'] == carrier_name]
    if not rows.empty:
        actual = rows['marginal_cost_eur_mwh'].iloc[0]
        status = "✓" if actual == expected_mc else "✗"
        print(f"  {status} {carrier_name}: {actual} €/MWh (expected {expected_mc})")


Created dataset with 69 technologies
0 technologies not found in costs CSV

Unique PyPSA carriers:
Total: 27 carriers
['battery', 'coal', 'coal-ccs', 'electrolyser', 'gas', 'gas-ccs', 'h2-ccgt', 'h2-fuel-cell', 'hydro-phs', 'hydro-phs-pure', 'hydro-pondage', 'hydro-reservoir', 'hydro-ror', 'lignite', 'lignite-ccs', 'nuclear', 'offwind', 'oil-heavy', 'oil-light', 'oil-shale', 'onwind', 'other-res', 'other-thermal', 'solar-pv-rooftop', 'solar-pv-utility', 'solar-thermal', 'solar-thermal-w-storage']

Hardcoded MC verification:
  ✓ other-res: 45.0 €/MWh (expected 45.0)


In [20]:
import sys
sys.path.insert(0, '.')
from helpers import CARRIERS

unique_set   = set(unique_carriers)
helpers_set  = set(CARRIERS)

only_in_unique  = sorted(unique_set  - helpers_set)
only_in_helpers = sorted(helpers_set - unique_set)
in_both         = sorted(unique_set  & helpers_set)

print(f"unique_carriers (pypsa_carrier from CSV) : {len(unique_set)}")
print(f"CARRIERS in helpers.py                   : {len(helpers_set)}")
print()
print(f"✓ In BOTH ({len(in_both)}):")
for c in in_both:
    print(f"   {c}")

print(f"\n⚠  Only in unique_carriers — NOT in CARRIERS ({len(only_in_unique)}):")
for c in only_in_unique:
    print(f"   {c}")

print(f"\n⚠  Only in CARRIERS — NOT in unique_carriers ({len(only_in_helpers)}):")
for c in only_in_helpers:
    print(f"   {c}")


unique_carriers (pypsa_carrier from CSV) : 27
CARRIERS in helpers.py                   : 28

✓ In BOTH (26):
   battery
   coal
   coal-ccs
   gas
   gas-ccs
   h2-ccgt
   h2-fuel-cell
   hydro-phs
   hydro-phs-pure
   hydro-pondage
   hydro-reservoir
   hydro-ror
   lignite
   lignite-ccs
   nuclear
   offwind
   oil-heavy
   oil-light
   oil-shale
   onwind
   other-res
   other-thermal
   solar-pv-rooftop
   solar-pv-utility
   solar-thermal
   solar-thermal-w-storage

⚠  Only in unique_carriers — NOT in CARRIERS (1):
   electrolyser

⚠  Only in CARRIERS — NOT in unique_carriers (2):
   chp
   uranium


In [21]:
import sys
sys.path.insert(0, '.')
from helpers import CARRIERS, CARRIERS_TO_SIMPLE

carriers_set = set(CARRIERS)
simple_keys  = set(CARRIERS_TO_SIMPLE.keys())

missing_in_simple = sorted(carriers_set - simple_keys)

print(f"Keys in CARRIERS but missing from CARRIERS_TO_SIMPLE ({len(missing_in_simple)}):")
for c in missing_in_simple:
    print(f"  {repr(c)}")


Keys in CARRIERS but missing from CARRIERS_TO_SIMPLE (1):
  'uranium'
